## Notebook 02 — RibFrac CT Preprocessing
<p> Patient-level splitting is performed using the same patient-level procedure established in Notebook 01 and is preserved throughout the preprocessing pipeline.</p>
<p> This notebook preprocesses RibFrac CT volumes and their corresponding fracture annotation volumes for the Enhanced 3D ResUNet framework.</p>
<p> The main goal of Notebook 02 is to transform each CT volume and its corresponding annotation into a consistent 3D representation suitable for the Enhanced 3D ResUNet.</p>

<h4>Processing includes:</h4>
<ol>
<li>GitHub repository and data-module setup</li>
<li>CT and label loading</li>
<li>Metadata loading</li>
<li>Patient-level train/validation/test split</li>
<li>Spatial validation</li>
<li>CT intensity clipping</li>
<li>CT intensity normalization</li>
<li>Spatial resampling</li>
<li>Label resampling using nearest-neighbor interpolation</li>
<li>Preprocessing validation</li>
<li>Saving the processed volumes</li>
</ol>


In [57]:
!git clone https://github.com/waboke/3D_ResUnet_Zenodo.git /kaggle/working/3D_ResUnet_Zenodo

fatal: destination path '/kaggle/working/3D_ResUnet_Zenodo' already exists and is not an empty directory.


In [58]:
!find /kaggle/working/3D_ResUnet_Zenodo "ribfrac_data_zenodo.py"

/kaggle/working/3D_ResUnet_Zenodo
/kaggle/working/3D_ResUnet_Zenodo/01_load_data_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/requirements.txt
/kaggle/working/3D_ResUnet_Zenodo/04_Model_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/06_Evaluation_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/__pycache__
/kaggle/working/3D_ResUnet_Zenodo/__pycache__/ribfrac_data_zenodo.cpython-312.pyc
/kaggle/working/3D_ResUnet_Zenodo/05_Training_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/README.md
/kaggle/working/3D_ResUnet_Zenodo/07_Visualization_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/ribfrac_data_zenodo.py
/kaggle/working/3D_ResUnet_Zenodo/02_preprocessing_zenodo.ipynb
/kaggle/working/3D_ResUnet_Zenodo/.git
/kaggle/working/3D_ResUnet_Zenodo/.git/refs
/kaggle/working/3D_ResUnet_Zenodo/.git/refs/remotes
/kaggle/working/3D_ResUnet_Zenodo/.git/refs/remotes/origin
/kaggle/working/3D_ResUnet_Zenodo/.git/refs/remotes/origin/HEAD
/kaggle/working/3D_ResUnet_Zenodo/.git/refs/heads
/kaggle/workin

In [59]:
from pathlib import Path

matches = list(
    Path("/kaggle").rglob("ribfrac_data_zenodo.py")
)

print("Files found:")

for path in matches:
    print(path)

Files found:
/kaggle/working/3D_ResUnet_Zenodo/ribfrac_data_zenodo.py


In [60]:
import sys

sys.path.insert(
    0,
    "/kaggle/working/3D_ResUnet_Zenodo"
)

from ribfrac_data_zenodo import (
    download_ribfrac_image,
    get_label_path,
)

print("ribfrac_data_zenodo imported successfully.")

ribfrac_data_zenodo imported successfully.


In [61]:
import ribfrac_data_zenodo

print("Module loaded from:")
print(ribfrac_data_zenodo.__file__)

print("\nFunctions/objects available:")
print(dir(ribfrac_data_zenodo))

Module loaded from:
/kaggle/working/3D_ResUnet_Zenodo/ribfrac_data_zenodo.py

Functions/objects available:
['BASE_DIR', 'CT_ZIP_URL', 'IMAGE_DIR', 'INFO_PATH', 'INFO_URL', 'LABEL_DIR', 'LABEL_ZIP_PATH', 'LABEL_ZIP_URL', 'METADATA_DIR', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_find_zip64_eocd', 'annotations', 'build_ct_zip_index', 'build_label_index', 'd', 'download_label_zip', 'download_metadata', 'download_ribfrac_image', 'get_ct_entry', 'get_label_path', 'get_patient_annotation_metadata', 'http_range_get', 'initialize_data_access', 'load_metadata', 'load_patient', 'nib', 'np', 'pd', 'requests', 'struct', 'validate_patient', 'zipfile', 'zlib']


In [62]:
## Import libraries
import os
import gc
import json
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from scipy.ndimage import zoom
from ribfrac_data_zenodo import (
    download_ribfrac_image,
    get_label_path,
    load_patient,
    validate_patient
)

print("RibFrac data module imported successfully.")

print("Libraries imported successfully.")

RibFrac data module imported successfully.
Libraries imported successfully.


In [63]:
#3. Define directories
## We will use the directories established in Notebook 01.

BASE_DIR = Path("/kaggle/working")

IMAGE_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"

PREPROCESSED_DIR = BASE_DIR / "preprocessed"

PREPROCESSED_IMAGE_DIR = PREPROCESSED_DIR / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_DIR / "labels"

PREPROCESSED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessing directories created:")
print(PREPROCESSED_IMAGE_DIR)
print(PREPROCESSED_LABEL_DIR)

Preprocessing directories created:
/kaggle/working/preprocessed/images
/kaggle/working/preprocessed/labels


In [64]:
# 4. — Check directories
print("Image directory exists:", IMAGE_DIR.exists())
print("Label directory exists:", LABEL_DIR.exists())

print("\nPreprocessed directories:")
print("Images:", PREPROCESSED_IMAGE_DIR)
print("Labels:", PREPROCESSED_LABEL_DIR)

Image directory exists: True
Label directory exists: True

Preprocessed directories:
Images: /kaggle/working/preprocessed/images
Labels: /kaggle/working/preprocessed/labels


In [70]:
# Cell 4A — Load RibFrac Metadata
# Download RibFrac metadata if it is not already available

from pathlib import Path
import requests

WORK_DIR = Path("/kaggle/working")
METADATA_DIR = WORK_DIR / "ribfrac_metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

INFO_PATH = METADATA_DIR / "ribfrac-train-info-1.csv"

INFO_URL = "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv"

if not INFO_PATH.exists():

    print("Metadata not found.")
    print("Downloading RibFrac metadata...")

    response = requests.get(INFO_URL, timeout=120)

    print("HTTP status:", response.status_code)

    response.raise_for_status()

    INFO_PATH.write_bytes(response.content)

    print("Metadata downloaded successfully.")

else:
    print("Metadata already exists.")

print("\nMetadata path:")
print(INFO_PATH)

print("File exists:", INFO_PATH.exists())
print("File size:", INFO_PATH.stat().st_size, "bytes")

Metadata not found.
HTTP status: 200
Metadata downloaded successfully.

Metadata path:
/kaggle/working/ribfrac_metadata/ribfrac-train-info-1.csv
File exists: True
File size: 48583 bytes


In [71]:
# Cell 5A — Read Metadata

info_df = pd.read_csv(INFO_PATH)

print("Metadata loaded successfully.")
print("Shape:", info_df.shape)

print("\nColumns:")
print(info_df.columns.tolist())

print("\nFirst 5 rows:")
display(info_df.head())

Metadata loaded successfully.
Shape: (3152, 3)

Columns:
['public_id', 'label_id', 'label_code']

First 5 rows:


,public_id,label_id,label_code
0,RibFrac1,0,0
1,RibFrac1,1,2
2,RibFrac1,2,2
3,RibFrac2,0,0
4,RibFrac2,1,-1


In [72]:
# Cell 6A — Create Patient-Level Split

from sklearn.model_selection import train_test_split

# Get unique patient IDs
patient_ids = sorted(info_df["public_id"].unique())

print("Total unique patients:", len(patient_ids))

# 80% development / 20% test
train_patients, test_patients = train_test_split(
    patient_ids,
    test_size=0.20,
    random_state=42
)

# 80% training / 20% validation from the development set
train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.20,
    random_state=42
)

print("\nPatient-level split:")
print("Train:", len(train_patients))
print("Validation:", len(val_patients))
print("Test:", len(test_patients))

Total unique patients: 300

Patient-level split:
Train: 192
Validation: 48
Test: 60


In [73]:
# Cell 7A — Verify Patient-Level Split

print("Train ∩ Validation:",
      set(train_patients) & set(val_patients))

print("Train ∩ Test:",
      set(train_patients) & set(test_patients))

print("Validation ∩ Test:",
      set(val_patients) & set(test_patients))

assert len(set(train_patients) & set(val_patients)) == 0
assert len(set(train_patients) & set(test_patients)) == 0
assert len(set(val_patients) & set(test_patients)) == 0

print("\nPatient-level leakage check: PASS")


Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()


In [74]:
#5. — Define preprocessing parameters
HU_MIN = -1000.0
HU_MAX = 400.0

TARGET_SPACING = (1.0, 1.0, 1.0)

print("HU range:", HU_MIN, "to", HU_MAX)
print("Target spacing:", TARGET_SPACING)

HU range: -1000.0 to 400.0
Target spacing: (1.0, 1.0, 1.0)


In [75]:
#6. — Load a patient
def load_patient_for_preprocessing(patient_id):
    """
    Load one patient's CT and label volumes.

    The CT retrieval function and label retrieval function
    are defined in Notebook 01.
    """

    ct_path = download_ribfrac_image(patient_id)
    label_path = get_label_path(patient_id)

    ct_img = nib.load(str(ct_path))
    label_img = nib.load(str(label_path))

    ct_volume = ct_img.get_fdata(dtype=np.float32)
    label_volume = label_img.get_fdata(dtype=np.float32)

    return {
        "patient_id": patient_id,
        "ct_path": ct_path,
        "label_path": label_path,
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume
    }

In [76]:
#7 — Spatial validation
def validate_spatial_alignment(ct_img, label_img):
    """
    Validate shape, voxel spacing, and affine alignment.
    """

    ct_shape = ct_img.shape
    label_shape = label_img.shape

    ct_spacing = ct_img.header.get_zooms()[:3]
    label_spacing = label_img.header.get_zooms()[:3]

    shape_match = ct_shape == label_shape
    spacing_match = np.allclose(ct_spacing, label_spacing, atol=1e-5)
    affine_match = np.allclose(ct_img.affine, label_img.affine, atol=1e-4)

    return {
        "shape_match": shape_match,
        "spacing_match": spacing_match,
        "affine_match": affine_match,
        "ct_shape": ct_shape,
        "label_shape": label_shape,
        "ct_spacing": ct_spacing,
        "label_spacing": label_spacing
    }

In [77]:
#8 — CT intensity preprocessing
def preprocess_ct_intensity(ct_volume,
                            hu_min=HU_MIN,
                            hu_max=HU_MAX):
    """
    Clip CT intensities and normalize to [0, 1].
    """

    ct_volume = np.asarray(ct_volume, dtype=np.float32)

    # Clip HU values
    ct_volume = np.clip(ct_volume, hu_min, hu_max)

    # Normalize to [0, 1]
    ct_volume = (ct_volume - hu_min) / (hu_max - hu_min)

    return ct_volume.astype(np.float32)

In [78]:
#9 — Resampling function
def calculate_zoom_factors(original_spacing, target_spacing):
    """
    Calculate zoom factors for spatial resampling.
    """

    original_spacing = np.asarray(original_spacing, dtype=np.float32)
    target_spacing = np.asarray(target_spacing, dtype=np.float32)

    return original_spacing / target_spacing

In [79]:
#10 — Resample CT
def resample_ct(ct_volume,
                original_spacing,
                target_spacing=TARGET_SPACING):

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_ct = zoom(
        ct_volume,
        zoom=zoom_factors,
        order=1
    )

    return resampled_ct.astype(np.float32)

In [80]:
#11 — Resample label
def resample_label(label_volume,
                   original_spacing,
                   target_spacing=TARGET_SPACING):

    zoom_factors = calculate_zoom_factors(
        original_spacing,
        target_spacing
    )

    resampled_label = zoom(
        label_volume,
        zoom=zoom_factors,
        order=0
    )

    return resampled_label.astype(np.int16)

In [81]:
#12 — Complete preprocessing function
def preprocess_patient(patient_id):

    print("=" * 60)
    print(f"Processing patient: {patient_id}")
    print("=" * 60)

    # -------------------------------------------------
    # 1. Load patient
    # -------------------------------------------------

    patient = load_patient_for_preprocessing(patient_id)

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct_volume = patient["ct_volume"]
    label_volume = patient["label_volume"]

    # -------------------------------------------------
    # 2. Validate spatial alignment
    # -------------------------------------------------

    validation = validate_spatial_alignment(
        ct_img,
        label_img
    )

    if not validation["shape_match"]:
        raise ValueError(
            f"CT/label shape mismatch for {patient_id}"
        )

    if not validation["spacing_match"]:
        raise ValueError(
            f"CT/label spacing mismatch for {patient_id}"
        )

    if not validation["affine_match"]:
        raise ValueError(
            f"CT/label affine mismatch for {patient_id}"
        )

    print("Spatial validation: PASS")

    # -------------------------------------------------
    # 3. Get original spacing
    # -------------------------------------------------

    original_spacing = ct_img.header.get_zooms()[:3]

    print("Original spacing:", original_spacing)

    # -------------------------------------------------
    # 4. CT intensity preprocessing
    # -------------------------------------------------

    ct_volume = preprocess_ct_intensity(ct_volume)

    print(
        "CT intensity range after normalization:",
        float(ct_volume.min()),
        "to",
        float(ct_volume.max())
    )

    # -------------------------------------------------
    # 5. Resample CT
    # -------------------------------------------------

    ct_resampled = resample_ct(
        ct_volume,
        original_spacing,
        TARGET_SPACING
    )

    # -------------------------------------------------
    # 6. Resample label
    # -------------------------------------------------

    label_resampled = resample_label(
        label_volume,
        original_spacing,
        TARGET_SPACING
    )

    # -------------------------------------------------
    # 7. Validate output
    # -------------------------------------------------

    print("Resampled CT shape:", ct_resampled.shape)
    print("Resampled label shape:", label_resampled.shape)

    print(
        "CT range:",
        float(ct_resampled.min()),
        "to",
        float(ct_resampled.max())
    )

    print(
        "Label values:",
        np.unique(label_resampled)
    )

    return {
        "patient_id": patient_id,
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": original_spacing,
        "target_spacing": TARGET_SPACING
    }

In [82]:
#13 — Test with RibFrac128
test_patient = preprocess_patient("RibFrac128")

Processing patient: RibFrac128
Spatial validation: PASS
Original spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))
CT intensity range after normalization: 0.0 to 1.0
Resampled CT shape: (395, 395, 421)
Resampled label shape: (395, 395, 421)
CT range: 0.0 to 1.0
Label values: [0 1 2 3 4 5 6 7 8]


In [83]:
#14 — Validate label values
unique_labels = np.unique(test_patient["label"])

print("Unique label values:")
print(unique_labels)

if np.all(unique_labels == unique_labels.astype(int)):
    print("Label integrity check: PASS")
else:
    print("Label integrity check: FAIL")

Unique label values:
[0 1 2 3 4 5 6 7 8]
Label integrity check: PASS


In [84]:
#15 — Check CT and label dimensions
ct_shape = test_patient["ct"].shape
label_shape = test_patient["label"].shape

print("CT shape:", ct_shape)
print("Label shape:", label_shape)

assert ct_shape == label_shape

print("CT/label shape check: PASS")

CT shape: (395, 395, 421)
Label shape: (395, 395, 421)
CT/label shape check: PASS


In [85]:
#16 — Save preprocessed patient
def save_preprocessed_patient(patient_data):

    patient_id = patient_data["patient_id"]

    ct = patient_data["ct"]
    label = patient_data["label"]

    ct_path = PREPROCESSED_IMAGE_DIR / f"{patient_id}-image.npy"
    label_path = PREPROCESSED_LABEL_DIR / f"{patient_id}-label.npy"

    np.save(ct_path, ct)
    np.save(label_path, label)

    print(f"Saved CT:    {ct_path}")
    print(f"Saved label: {label_path}")

    return ct_path, label_path

In [86]:
#17 — Preprocess and save RibFrac128

patient_id = "RibFrac128"

patient_data = preprocess_patient(patient_id)

ct_path, label_path = save_preprocessed_patient(patient_data)

print("Saved CT:", ct_path)
print("Saved label:", label_path)

Processing patient: RibFrac128
Spatial validation: PASS
Original spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))
CT intensity range after normalization: 0.0 to 1.0
Resampled CT shape: (395, 395, 421)
Resampled label shape: (395, 395, 421)
CT range: 0.0 to 1.0
Label values: [0 1 2 3 4 5 6 7 8]
Saved CT:    /kaggle/working/preprocessed/images/RibFrac128-image.npy
Saved label: /kaggle/working/preprocessed/labels/RibFrac128-label.npy
Saved CT: /kaggle/working/preprocessed/images/RibFrac128-image.npy
Saved label: /kaggle/working/preprocessed/labels/RibFrac128-label.npy


In [89]:
#18 — Verify saved files
saved_ct = np.load(ct_path)
saved_label = np.load(label_path)

print("Saved CT shape:", saved_ct.shape)
print("Saved label shape:", saved_label.shape)

print("Saved CT dtype:", saved_ct.dtype)
print("Saved label dtype:", saved_label.dtype)

Saved CT shape: (395, 395, 421)
Saved label shape: (395, 395, 421)
Saved CT dtype: float32
Saved label dtype: int16


In [90]:
#19 — Check memory
print(
    "CT memory:",
    saved_ct.nbytes / (1024 ** 2),
    "MB"
)

print(
    "Label memory:",
    saved_label.nbytes / (1024 ** 2),
    "MB"
)

CT memory: 250.5742073059082 MB
Label memory: 125.2871036529541 MB


In [91]:
#20 — Release test patient
del test_patient
del saved_ct
del saved_label

gc.collect()

print("Temporary arrays released from memory.")

Temporary arrays released from memory.


In [92]:
#21 — Automatic preprocessing of the training set
preprocess_patient("RibFrac128")

Processing patient: RibFrac128
Spatial validation: PASS
Original spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))
CT intensity range after normalization: 0.0 to 1.0
Resampled CT shape: (395, 395, 421)
Resampled label shape: (395, 395, 421)
CT range: 0.0 to 1.0
Label values: [0 1 2 3 4 5 6 7 8]


{'patient_id': 'RibFrac128',
 'ct': array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        ...,
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
     

## Training-set preprocessing

The complete training-set loop is kept disabled until the single-patient pipeline has been verified. This prevents an accidental download/resampling of all 192 training patients.


In [ ]:
# Cell 21 — Optional training-set preprocessing

PROCESS_TRAINING_SET = False

if PROCESS_TRAINING_SET:

    for i, patient_id in enumerate(train_patients, start=1):

        print(f"\nPatient {i}/{len(train_patients)}")
        print(f"Patient ID: {patient_id}")

        processed = preprocess_patient(patient_id)

        save_preprocessed_patient(processed)

        del processed
        gc.collect()

    print("\nTraining-set preprocessing completed.")

else:

    print(
        "Training-set preprocessing is disabled. "
        "Set PROCESS_TRAINING_SET = True only after "
        "RibFrac128 has been verified successfully."
    )


In [ ]:
# Cell 22 — Final preprocessing directory check

print("Preprocessed image directory:")
print(PREPROCESSED_IMAGE_DIR)

print("\nPreprocessed label directory:")
print(PREPROCESSED_LABEL_DIR)

print("\nNumber of saved CT files:",
      len(list(PREPROCESSED_IMAGE_DIR.glob("*.npy"))))

print("Number of saved label files:",
      len(list(PREPROCESSED_LABEL_DIR.glob("*.npy"))))


### Notebook 02 status

The single-patient preprocessing pipeline is complete when RibFrac128 passes loading, spatial validation, preprocessing, label-integrity validation, saving, and saved-file verification.

After this is confirmed, enable `PROCESS_TRAINING_SET` and process the training patients sequentially.
